In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Verify everything is working
print(f"PyTorch version: {torch.__version__}")
print("Ready to build neural networks!")

PyTorch version: 2.10.0
Ready to build neural networks!


In [2]:
# Prepare the Iris data
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Convert to PyTorch tensors
# (PyTorch works with "tensors" instead of numpy arrays)
X_train_t = torch.FloatTensor(X_train)
X_test_t  = torch.FloatTensor(X_test)
y_train_t = torch.LongTensor(y_train)
y_test_t  = torch.LongTensor(y_test)

# Define the neural network
class IrisNet(nn.Module):
    def __init__(self):
        super(IrisNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 16),   # 4 inputs → 16 neurons
            nn.ReLU(),           # activation function
            nn.Linear(16, 8),   # 16 neurons → 8 neurons
            nn.ReLU(),
            nn.Linear(8, 3)     # 8 neurons → 3 outputs (species)
        )
    
    def forward(self, x):
        return self.network(x)

model = IrisNet()
print("Neural network architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

Neural network architecture:
IrisNet(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=3, bias=True)
  )
)

Total parameters: 243


In [3]:
# Train the neural network
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 100
losses = []

for epoch in range(epochs):
    # Forward pass
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    # Backward pass (this is where backpropagation happens)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")

# Evaluate
with torch.no_grad():
    outputs = model(X_test_t)
    predictions = torch.argmax(outputs, dim=1)
    accuracy = (predictions == y_test_t).float().mean()
    print(f"\nTest accuracy: {accuracy:.2%}")

Epoch 20/100 - Loss: 0.4473
Epoch 40/100 - Loss: 0.1337
Epoch 60/100 - Loss: 0.0605
Epoch 80/100 - Loss: 0.0475
Epoch 100/100 - Loss: 0.0448

Test accuracy: 100.00%


In [6]:
def fgsm_attack(model, X, y, epsilon):
    """
    FGSM - Fast Gradient Sign Method
    epsilon controls attack strength (bigger = stronger attack)
    """
    X_attack = X.clone().requires_grad_(True)
    
    # Forward pass
    outputs = model(X_attack)
    loss = criterion(outputs, y)
    
    # Backward pass - calculate gradients with respect to INPUT
    model.zero_grad()
    loss.backward()
    
    # The attack - nudge input in the direction that maximises loss
    perturbation = epsilon * X_attack.grad.sign()
    X_adversarial = X_attack + perturbation
    
    return X_adversarial.detach()

# Test at different attack strengths
print("FGSM Attack Results:")
print(f"{'Epsilon':<12} {'Accuracy':<12} {'Drop'}")
print("-" * 36)

# Baseline with no attack
with torch.no_grad():
    outputs = model(X_test_t)
    predictions = torch.argmax(outputs, dim=1)
    baseline = (predictions == y_test_t).float().mean().item()

print(f"{'No attack':<12} {baseline:.2%}")

for epsilon in [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]:
    X_adv = fgsm_attack(model, X_test_t, y_test_t, epsilon)
    
    with torch.no_grad():
        outputs = model(X_adv)
        predictions = torch.argmax(outputs, dim=1)
        accuracy = (predictions == y_test_t).float().mean().item()
        drop = baseline - accuracy
    
    print(f"{epsilon:<12} {accuracy:<12.2%} -{drop:.2%}")

FGSM Attack Results:
Epsilon      Accuracy     Drop
------------------------------------
No attack    100.00%
0.01         100.00%      -0.00%
0.05         96.67%       -3.33%
0.1          93.33%       -6.67%
0.2          86.67%       -13.33%
0.3          76.67%       -23.33%
0.5          46.67%       -53.33%


In [7]:
# Train a new model with adversarial examples mixed in
defended_model = IrisNet()
optimizer_d = optim.Adam(defended_model.parameters(), lr=0.01)

epochs = 150

for epoch in range(epochs):
    # Train on clean data
    outputs = defended_model(X_train_t)
    loss_clean = criterion(outputs, y_train_t)
    
    # Generate adversarial examples during training
    X_adv_train = fgsm_attack(defended_model, X_train_t, y_train_t, epsilon=0.2)
    
    # Train on adversarial data too
    outputs_adv = defended_model(X_adv_train)
    loss_adv = criterion(outputs_adv, y_train_t)
    
    # Combined loss - half clean, half adversarial
    loss = 0.5 * loss_clean + 0.5 * loss_adv
    
    optimizer_d.zero_grad()
    loss.backward()
    optimizer_d.step()

# Compare original vs defended model under attack
print(f"{'Epsilon':<10} {'Original':<14} {'Defended':<14} {'Improvement'}")
print("-" * 52)

for epsilon in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]:
    # Original model
    if epsilon == 0:
        with torch.no_grad():
            out = model(X_test_t)
            acc_orig = (torch.argmax(out, dim=1) == y_test_t).float().mean().item()
            out_d = defended_model(X_test_t)
            acc_def = (torch.argmax(out_d, dim=1) == y_test_t).float().mean().item()
    else:
        X_adv = fgsm_attack(model, X_test_t, y_test_t, epsilon)
        with torch.no_grad():
            out = model(X_adv)
            acc_orig = (torch.argmax(out, dim=1) == y_test_t).float().mean().item()
        
        X_adv_d = fgsm_attack(defended_model, X_test_t, y_test_t, epsilon)
        with torch.no_grad():
            out_d = defended_model(X_adv_d)
            acc_def = (torch.argmax(out_d, dim=1) == y_test_t).float().mean().item()
    
    improvement = acc_def - acc_orig
    print(f"{epsilon:<10} {acc_orig:<14.2%} {acc_def:<14.2%} +{improvement:.2%}")

Epsilon    Original       Defended       Improvement
----------------------------------------------------
0.0        100.00%        100.00%        +0.00%
0.05       96.67%         96.67%         +0.00%
0.1        93.33%         93.33%         +0.00%
0.2        86.67%         83.33%         +-3.33%
0.3        76.67%         80.00%         +3.33%
0.5        46.67%         66.67%         +20.00%


In [8]:
def input_smoothing(X, noise_factor=0.05):
    """
    Input smoothing defence - adds tiny random noise to blur
    the sharp perturbations that FGSM adds
    """
    noise = torch.randn_like(X) * noise_factor
    return X + noise

# Test input smoothing on top of the defended model
print(f"{'Epsilon':<10} {'No defence':<14} {'Adv training':<16} {'Both combined'}")
print("-" * 58)

for epsilon in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]:
    # No defence
    if epsilon == 0:
        with torch.no_grad():
            out = model(X_test_t)
            acc_none = (torch.argmax(out, dim=1) == y_test_t).float().mean().item()
    else:
        X_adv = fgsm_attack(model, X_test_t, y_test_t, epsilon)
        with torch.no_grad():
            out = model(X_adv)
            acc_none = (torch.argmax(out, dim=1) == y_test_t).float().mean().item()

    # Adversarial training only
    if epsilon == 0:
        with torch.no_grad():
            out_d = defended_model(X_test_t)
            acc_adv_train = (torch.argmax(out_d, dim=1) == y_test_t).float().mean().item()
    else:
        X_adv_d = fgsm_attack(defended_model, X_test_t, y_test_t, epsilon)
        with torch.no_grad():
            out_d = defended_model(X_adv_d)
            acc_adv_train = (torch.argmax(out_d, dim=1) == y_test_t).float().mean().item()

    # Both combined - smoothing + adversarial training
    if epsilon == 0:
        with torch.no_grad():
            X_smooth = input_smoothing(X_test_t)
            out_both = defended_model(X_smooth)
            acc_both = (torch.argmax(out_both, dim=1) == y_test_t).float().mean().item()
    else:
        X_adv_d = fgsm_attack(defended_model, X_test_t, y_test_t, epsilon)
        with torch.no_grad():
            X_smooth = input_smoothing(X_adv_d)
            out_both = defended_model(X_smooth)
            acc_both = (torch.argmax(out_both, dim=1) == y_test_t).float().mean().item()

    print(f"{epsilon:<10} {acc_none:<14.2%} {acc_adv_train:<16.2%} {acc_both:.2%}")

Epsilon    No defence     Adv training     Both combined
----------------------------------------------------------
0.0        100.00%        100.00%          100.00%
0.05       96.67%         96.67%           96.67%
0.1        93.33%         93.33%           93.33%
0.2        86.67%         83.33%           80.00%
0.3        76.67%         80.00%           80.00%
0.5        46.67%         66.67%           66.67%
